# v27 XY近傍版 — 同一typewellでなく「XY距離が近い坑井」を参照に pseudo-typewell PF混合

## 仮説
同一typewell(全体の4.4%)は対象が少ない。代わりに **horizontal の XY 座標が近い坑井**を「同じ地質を共有する参照」とみなせば、
**ほぼ全坑井**で pseudo-typewell 混合(v27の勝ち筋)が使えるはず。

## 変更点(同一typewell版との差)
- 参照選択を **XY重心距離の近傍・最大 `K_MAX` 本・`MAX_DIST` 以内** に変更(同一typewell条件を撤廃)。
- **遠すぎる坑井は選ばない**(`MAX_DIST`)。**近傍0本の坑井は対象外**(選択肢ゼロ)。
- PF/pseudo-typewell混合/pool/ゲートのエンジンは同一typewell版と**完全に同じ**(gridに入れる参照が変わるだけ)。

## 較正
同一typewell群のXY重心距離: median≈4800 / p90≈9500 / max≈14300 → `MAX_DIST=5000` を既定(同地質スケール)。
`MAX_DIST=5000, K_MAX=5` で近傍≥1本= **766/773 坑井**(0本=7坑井のみ)。=**ほぼ全坑井で検証可能**。

## 指標
overall RMSE(eval行プール)。valid=各坑井, 参照=そのXY近傍(他坑井, 全真値既知)。OOF(自分は参照に入らない)。


In [1]:
# ===== import + データパス自動判定 + train坑井一覧 =====
import os, glob, json, time, hashlib, warnings
import numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore"); np.seterr(all="ignore")

CANDS = [
    "/kaggle/input/rogii-wellbore-geology-prediction",
    "/kaggle/input/competitions/rogii-wellbore-geology-prediction",
    r"C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction",
]
COMP = next((p for p in CANDS if os.path.exists(os.path.join(p, "sample_submission.csv"))), None)
if COMP is None:
    hits = glob.glob("/kaggle/input/**/sample_submission.csv", recursive=True)
    COMP = os.path.dirname(hits[0]) if hits else CANDS[-1]
COMP = Path(COMP); TRAIN = COMP / "train"
ART = Path(r"C:\Users\kosaka256\Documents\rogii_claude\v27\artifacts_v27"); ART.mkdir(parents=True, exist_ok=True)

WELLS = sorted(os.path.basename(p).split("__")[0]
               for p in glob.glob(str(TRAIN / "*__horizontal_well.csv")))
print("COMP:", COMP, "| train wells:", len(WELLS), "| ART:", ART)

COMP: C:\Users\kosaka256\Documents\rogii_claude\rogii-wellbore-geology-prediction | train wells: 773 | ART: C:\Users\kosaka256\Documents\rogii_claude\v27\artifacts_v27


In [ ]:
# ===== 各坑井の XY 代表点(重心) + 近傍探索(最大K本, MAX_DIST以内) =====
import hashlib
from collections import defaultdict
K_MAX    = 3        # 近傍の最大本数
MAX_DIST = 2000.0   # これより遠い坑井は参照にしない(同一typewell群XY距離 median≈4800)

# typewell(build_ctx用) + XY重心
TW_CACHE = {}; CENT = {}
t0 = time.time()
for k, wid in enumerate(WELLS):
    hw = pd.read_csv(TRAIN / f"{wid}__horizontal_well.csv", usecols=["X", "Y"])
    CENT[wid] = (float(hw["X"].mean()), float(hw["Y"].mean()))
    TW_CACHE[wid] = pd.read_csv(TRAIN / f"{wid}__typewell.csv")
    if (k + 1) % 200 == 0: print(f"  loaded {k+1}/{len(WELLS)} ({time.time()-t0:.0f}s)")
names = list(WELLS); XY = np.array([CENT[w] for w in names])
D = np.hypot(XY[:, 0][:, None] - XY[:, 0][None, :], XY[:, 1][:, None] - XY[:, 1][None, :])
np.fill_diagonal(D, np.inf)

# 較正(参考): 同一typewell群のXY距離
fp = {w: hashlib.sha1(TW_CACHE[w].to_csv(index=False).encode()).hexdigest()[:16] for w in names}
grp = defaultdict(list)
for w in names: grp[fp[w]].append(w)
intra = np.array([np.hypot(CENT[a][0]-CENT[b][0], CENT[a][1]-CENT[b][1])
                  for v in grp.values() if len(v) >= 2 for ai, a in enumerate(v) for b in v[ai+1:]])
if len(intra):
    print("同一typewell群XY距離: median=%.0f p90=%.0f max=%.0f (MAX_DIST=%.0fの参考)" %
          (np.median(intra), np.percentile(intra, 90), intra.max(), MAX_DIST))

# 近傍選択: 最大K本 かつ MAX_DIST 以内
NBR = {}
for i, w in enumerate(names):
    order = np.argsort(D[i]); order = order[D[i][order] <= MAX_DIST][:K_MAX]
    NBR[w] = [(names[j], float(D[i][j])) for j in order]
nn = np.array([len(v) for v in NBR.values()])
print("MAX_DIST=%.0f K_MAX=%d | 近傍数 median=%d max=%d" % (MAX_DIST, K_MAX, int(np.median(nn)), int(nn.max())))
print("近傍0本(対象外)= %d 本 / 対象(>=1本)= %d 本" % (int((nn == 0).sum()), int((nn >= 1).sum())))

In [ ]:
# ===== 各坑井を valid、そのXY近傍を参照にした OOF ペア(近傍0本は除外) =====
oof_pairs = []
for w in names:
    refs = [r for r, d in NBR[w]]
    if not refs: continue          # 選択肢ゼロは対象外
    oof_pairs.append(dict(valid=w, refs=refs, n_refs=len(refs),
                          group_size=len(refs) + 1,      # c11の互換フィールド
                          nn_dist=NBR[w][0][1], far_dist=NBR[w][-1][1]))
print("OOF valid坑井(近傍>=1本):", len(oof_pairs), "/", len(names))
pairs_df = pd.DataFrame([{"valid": p["valid"], "n_refs": p["n_refs"],
                          "nn_dist": round(p["nn_dist"]), "far_dist": round(p["far_dist"])} for p in oof_pairs])
print("参照本数の分布:"); print(pairs_df["n_refs"].value_counts().sort_index().to_string())
pairs_df.to_csv(ART / "xyknn_oof_pairs.csv", index=False)
print("saved:", ART / "xyknn_oof_pairs.csv")

In [ ]:
# ===== データアクセサ + 整合性チェック =====
# valid坑井: test模擬(GR全区間 + TVT_input既知 + eval区間の隠し真値TVT)
# 参照坑井 : train全既知模擬(GR + TVT全真値)
_HW_CACHE = {}
def _load_hw(wid):
    if wid not in _HW_CACHE:
        _HW_CACHE[wid] = pd.read_csv(TRAIN / f"{wid}__horizontal_well.csv")
    return _HW_CACHE[wid]

def ref_arrays(wid):
    hw = _load_hw(wid)
    return dict(wid=wid, md=hw["MD"].to_numpy(float), x=hw["X"].to_numpy(float),
                y=hw["Y"].to_numpy(float), z=hw["Z"].to_numpy(float),
                gr=hw["GR"].to_numpy(float), tvt=hw["TVT"].to_numpy(float))   # 全真値

def valid_arrays(wid):
    hw = _load_hw(wid)
    ev = hw["TVT_input"].isna().to_numpy()
    last_kn = hw.loc[hw["TVT_input"].notna(), "TVT_input"]
    return dict(wid=wid, md=hw["MD"].to_numpy(float), x=hw["X"].to_numpy(float),
                y=hw["Y"].to_numpy(float), z=hw["Z"].to_numpy(float),
                gr=hw["GR"].to_numpy(float),
                tvt_input=hw["TVT_input"].to_numpy(float),   # 既知prefix(eval=NaN)
                tvt_true=hw["TVT"].to_numpy(float),          # 採点用の隠し真値
                eval_mask=ev,
                last_known_tvt=float(last_kn.iloc[-1]) if len(last_kn) else np.nan,
                ids=[f"{wid}_{i}" for i in range(len(hw))])

# --- サンプル確認 ---
_p = oof_pairs[0]; _va = valid_arrays(_p["valid"])
print("sample valid:", _p["valid"], "| n_refs", _p["n_refs"], "| nn_dist", round(_p["nn_dist"]), "| far", round(_p["far_dist"]))
print("  eval行", int(_va["eval_mask"].sum()), "| 既知prefix", int((~_va["eval_mask"]).sum()), "| last_known", round(_va["last_known_tvt"], 2))

In [ ]:
# ===== 参照プール + カバレッジ・ゲート + M1: 構造 datum 転写 =====
# 発想: 同一typewell=同一ブロック。参照train坑井は U=TVT+Z(構造datum)の傾斜を真値で持つ。
# 参照の U(MD) を valid の MD へ補間し、valid の既知prefixで定数オフセットを較正→eval を予測。
# = PFのdip外挿を「隣接坑井の既知傾斜」で置換。参照MD外/参照TVT範囲外(トゥ)は NaN(=フォールバック)。
def _ref_pool_tvt(refs):
    rt = np.concatenate([r["tvt"] for r in refs]); rt = rt[np.isfinite(rt)]
    return float(np.min(rt)), float(np.max(rt))

def match_m1(va, refs, deg=0, min_prefix=20, tvt_margin=2.0):
    # ★実験結果(2026-07): deg0 は 10本群でまぐれ勝ちする坑井もあるが、10群プールでは hold 22.3 < M1 32.6 で悪化。
    #   同一typewell=同一「層序(GR-TVT)」であって同一「dip/幾何」ではない → MD整列は水平区間で dip差を積分し破綻。
    #   結論: この幾何ベース転写は非推奨。GR(層序)空間の照合(M2/PF augmented likelihood)へ。ここは比較用に残す。
    md = va["md"]; z = va["z"]; ev = va["eval_mask"]
    U_known = va["tvt_input"] + z                      # 既知U(eval=NaN)
    preds = []
    for r in refs:
        rmd = r["md"]; rU = r["tvt"] + r["z"]
        o = np.argsort(rmd); rmd = rmd[o]; rU = rU[o]
        keep = np.concatenate([[True], np.diff(rmd) > 0])   # 単調化(補間用)
        rmd = rmd[keep]; rU = rU[keep]
        Ui = np.interp(md, rmd, rU, left=np.nan, right=np.nan)  # 参照MD外=NaN
        kn = (~ev) & np.isfinite(U_known) & np.isfinite(Ui)
        if int(kn.sum()) < min_prefix: continue
        if deg == 0:
            corr = np.full(len(md), float(np.median(U_known[kn] - Ui[kn])))  # 定数較正
        else:
            corr = np.polyval(np.polyfit(md[kn], U_known[kn] - Ui[kn], deg), md)  # 1次=dip差も較正
        preds.append(Ui + corr)
    n_ev = int(ev.sum())
    if not preds: return np.full(n_ev, np.nan)
    Upred = np.nanmedian(np.vstack(preds), axis=0)      # 複数参照はU空間で中央値(平均は多峰で危険)
    pe = (Upred - z)[ev].astype(float)
    rtmin, rtmax = _ref_pool_tvt(refs)
    pe[(pe < rtmin - tvt_margin) | (pe > rtmax + tvt_margin)] = np.nan   # トゥ(範囲外)→NaN
    return pe

def gated(matcher, **kw):
    """matcher の NaN(トゥ/未カバー)を hold で埋めた完成predictorを返す(本番はPFにフォールバック)。"""
    def f(va, refs):
        pe = np.asarray(matcher(va, refs, **kw), float).copy()
        bad = ~np.isfinite(pe)
        if bad.any(): pe[bad] = va["last_known_tvt"]
        return pe
    return f
print("M1(構造datum転写) ready")

In [ ]:
# ===== PF エンジン(v25 r0 torch/GPU 忠実移植) + 尤度grid(typewell / +疑似typewell) =====
# v25 submit_smoother の r0 torch エンジンをそのまま持込み(numpy移植でなくGPUバッチ=v25と等価)。
# grid(gg) を typewell / 混合(疑似typewell) で差し替えて A/B。出力は固定ラグ平滑(SMOOTH_LAG)。
import torch, time
V27_DEV = "cuda" if torch.cuda.is_available() else "cpu"; V27_DT = torch.float32
PF_N       = 400
N_SEED     = 32
SMOOTH_LAG = 192     # v25 6.786 と同じ固定ラグ。0=forward PF
MIX_W      = 1.0     # 疑似typewell混合比(covered帯; 1.0=実HWテンプレで完全置換=勝者, 0=typewellのみ)
POOL_MODE  = "median" # 複数参照の疑似typewell生成: "mean" / "median"(平均のボケ回避) / "best1"(最類似1本)
MIX_MAX_REFS = 1      # 混合を適用する参照本数の上限(1=ペアのみ。参照多数群はtypewellが強いので混ぜない=ゲート)
SEED       = 0
CHUNK      = 16      # GPUバッチ坑井数(OOMなら下げる)
print("v27 PF device:", V27_DEV, "| torch:", torch.__version__)

R0 = {
    "n_particles": 400, "likelihood_scale": 10.536319667317654,
    "init_pos_std": 0.9331541486020234, "init_rate_std": 0.00047753793163756304,
    "mom": 0.9997911117425017, "vn": 0.0006297672017716754, "pn": 0.005014009140727263,
    "rp": 0.7627309890952053, "rr": 0.00045663430474523125, "resamp": 0.032049849143995945,
    "gr_sig_min": 11.865793233968713, "gr_sig_max": 44.98927964612316, "gr_sig_def": 119.25397464636981,
    "gr_sig_mult": 3.058318309943017, "tvt_clip_margin": 150.56111180234504, "rate_clip": 2.0,
    "gr_power": 1.7826896851395453, "hgr_smooth_r": 0, "tw_gr_smooth_r": 0,
    "jump_prob": 0.002, "jump_std": 5.0, "phys_sig": 0.013666550145707291, "use_phys": False,
}
P0 = {**R0, "n_particles": PF_N, "smooth_lag": SMOOTH_LAG}

def _smooth(vals, fb, r):
    r = int(r); s = pd.Series(vals, dtype="float32").interpolate(limit_direction="both").fillna(float(fb))
    return s.to_numpy(np.float32) if r <= 0 else s.rolling(2*r+1, center=True, min_periods=1).mean().to_numpy(np.float32)

def _grid(tw_tvt, tw_gr, step=0.2):
    a, b = float(tw_tvt.min()), float(tw_tvt.max()); g = np.arange(a, b+step, step)
    return np.interp(g, tw_tvt, tw_gr).astype(np.float64), float(a), float(step)

def _pf_smoother_core(md, z, gr, valid, grid, glen, vmin, step, gs, ls, ir, P, N, device, gen, dtype=V27_DT):
    B, T = md.shape
    ALPHA = P["mom"]; RN = P["vn"]; PN = P["pn"]; IR = P["init_rate_std"]; IS = P["init_pos_std"]
    RP = P["rp"]; RR = P["rr"]; RESAMP = P["resamp"]; RATE_CLIP = P["rate_clip"]
    GR_POWER = P["gr_power"]; JUMP_PROB = P["jump_prob"]; JUMP_STD = P["jump_std"]; CLIP = P["tvt_clip_margin"]
    PHYS_SIG = P["phys_sig"]; USE_P = P["use_phys"]
    L = int(P.get("smooth_lag", 32)); L = max(0, min(L, T))
    def rn(): return torch.randn((B, N), generator=gen, device=device, dtype=dtype)
    pos = ls[:, None] + IS*rn(); rate = ir[:, None] + IR*rn()
    w = torch.full((B, N), 1.0/N, device=device, dtype=dtype)
    log_lik = torch.zeros(B, device=device, dtype=torch.float64)
    pts_f = torch.zeros((B, T), device=device, dtype=dtype)
    pts_s = torch.zeros((B, T), device=device, dtype=dtype)
    tvt_lo = vmin - CLIP; tvt_hi = vmin + (glen.to(dtype)-1)*step + CLIP
    glast = torch.gather(grid, 1, (glen-1).clamp_min(0).unsqueeze(1)); g0 = grid[:, 0:1]
    pm = md[:, 0] - 1.0; arN = torch.arange(N, device=device, dtype=dtype)
    zero = torch.zeros((), device=device, dtype=dtype); dipp = ir[:, None]
    use_sm = L > 0
    if use_sm:
        buf = torch.full((B, L, N), float("nan"), device=device, dtype=dtype); ptr = 0
    for i in range(T):
        act = valid[:, i]; cur = md[:, i]; dm = (cur-pm).clamp_min(1.0)
        rate_n = ALPHA*rate + RN*rn()
        if RATE_CLIP > 0: rate_n = rate_n.clamp(-RATE_CLIP, RATE_CLIP)
        pos_n = pos + rate_n*dm[:, None] + PN*rn()
        if JUMP_PROB > 0 and JUMP_STD > 0:
            jm = torch.rand((B, N), generator=gen, device=device, dtype=dtype) < JUMP_PROB
            pos_n = pos_n + torch.where(jm, JUMP_STD*rn(), zero)
        zi = z[:, i][:, None]; tvt = (pos_n-zi).clamp(tvt_lo[:, None], tvt_hi[:, None]); pos_n = tvt+zi
        am = act[:, None]; pos = torch.where(am, pos_n, pos); rate = torch.where(am, rate_n, rate)
        gri = gr[:, i]; obs = act & ~torch.isnan(gri)
        v = pos-zi; ii = (v-vmin[:, None])/step[:, None]; i0 = torch.floor(ii).long()
        below = i0 < 0; above = i0 >= (glen-1)[:, None]
        i0c = i0.clamp_min(0); i0c = torch.minimum(i0c, (glen-2).clamp_min(0)[:, None])
        t = ii-i0c.to(dtype); gA = torch.gather(grid, 1, i0c); gB = torch.gather(grid, 1, i0c+1)
        eg = gA*(1-t)+gB*t; eg = torch.where(below, g0, eg); eg = torch.where(above, glast, eg)
        d = ((gri[:, None]-eg)/gs[:, None]).abs(); dp = d**GR_POWER
        lk = torch.where(dp < 600.0, torch.exp(-0.5*dp), zero).clamp_min(1e-300)
        avg = (w*lk).sum(1)
        log_lik = log_lik + torch.where(obs, torch.log(avg.clamp_min(1e-300).double()),
                                        torch.zeros((), device=device, dtype=torch.float64))
        if USE_P:
            dphys = (rate-dipp)/PHYS_SIG; lkp = torch.exp(-0.5*dphys*dphys).clamp_min(1e-300)
        else:
            lkp = torch.ones_like(w)
        w_new = torch.where(obs[:, None], w*lk*lkp, w*lkp)
        ws = w_new.sum(1, keepdim=True)
        w = torch.where(ws > 0, w_new/ws.clamp_min(1e-300), torch.full_like(w, 1.0/N))
        ne = (w*w).sum(1); need = act & ((1.0/ne) < (RESAMP*N))
        if bool(need.any()):
            cumw = torch.cumsum(w, 1); u0 = torch.rand((B, 1), generator=gen, device=device, dtype=dtype)*(1.0/N)
            u = u0 + arN[None, :]/N; idx = torch.searchsorted(cumw, u, right=False).clamp(0, N-1)
            pos_rs = torch.gather(pos, 1, idx) + RP*rn(); rate_rs = torch.gather(rate, 1, idx) + RR*rn()
            if RATE_CLIP > 0: rate_rs = rate_rs.clamp(-RATE_CLIP, RATE_CLIP)
            nm = need[:, None]; pos = torch.where(nm, pos_rs, pos); rate = torch.where(nm, rate_rs, rate)
            w = torch.where(nm, torch.full_like(w, 1.0/N), w)
            if use_sm:
                buf_g = torch.gather(buf, 2, idx[:, None, :].expand(B, L, N))
                buf = torch.where(need[:, None, None], buf_g, buf)
        pts_f[:, i] = (w*(pos-zi)).sum(1)
        if use_sm:
            if i >= L:
                old = buf[:, ptr, :]; pts_s[:, i-L] = (w*old).sum(1) - z[:, i-L]
            buf[:, ptr, :] = pos; ptr = (ptr+1) % L
        pm = torch.where(act, cur, pm)
    if use_sm:
        for j in range(max(0, T-L), T):
            pts_s[:, j] = (w*buf[:, j % L, :]).sum(1) - z[:, j]
    else:
        pts_s = pts_f
    return pts_f, pts_s, log_lik

def _pad(inps, device, dtype=V27_DT):
    W = len(inps); Tmax = max(len(x["md"]) for x in inps); Gmax = max(len(x["gg"]) for x in inps)
    md = torch.zeros((W, Tmax), dtype=dtype); z = torch.zeros((W, Tmax), dtype=dtype)
    gr = torch.full((W, Tmax), float("nan"), dtype=dtype); valid = torch.zeros((W, Tmax), dtype=torch.bool)
    grid = torch.zeros((W, Gmax), dtype=dtype); glen = torch.zeros(W, dtype=torch.long)
    vmin = torch.zeros(W, dtype=dtype); step = torch.zeros(W, dtype=dtype); gs = torch.zeros(W, dtype=dtype)
    ls = torch.zeros(W, dtype=dtype); ir = torch.zeros(W, dtype=dtype)
    for b, x in enumerate(inps):
        Tn = len(x["md"]); G = len(x["gg"])
        md[b, :Tn] = torch.from_numpy(x["md"].astype("float32"))
        if Tn < Tmax: md[b, Tn:] = md[b, Tn-1]
        z[b, :Tn] = torch.from_numpy(x["z"].astype("float32"))
        gr[b, :Tn] = torch.from_numpy(x["gr"].astype("float32")); valid[b, :Tn] = True
        grid[b, :G] = torch.from_numpy(x["gg"].astype("float32"))
        if G < Gmax: grid[b, G:] = grid[b, G-1]
        glen[b] = G; vmin[b] = x["gmin"]; step[b] = x["gst"]; gs[b] = x["gs"]; ls[b] = x["ls"]; ir[b] = x["ir"]
    to = lambda tt: tt.to(device)
    return dict(md=to(md), z=to(z), gr=to(gr), valid=to(valid), grid=to(grid), glen=to(glen),
                vmin=to(vmin), step=to(step), gs=to(gs), ls=to(ls), ir=to(ir))

def run_smoother_ens(inps, P, device, seed, n_seeds):
    N = int(P["n_particles"]); S = n_seeds; W = len(inps); pad = _pad(inps, device)
    rep = lambda tt: tt.repeat_interleave(S, 0); gen = torch.Generator(device=device); gen.manual_seed(seed)
    pf, ps, ll = _pf_smoother_core(
        rep(pad["md"]), rep(pad["z"]), rep(pad["gr"]), rep(pad["valid"]), rep(pad["grid"]),
        pad["glen"].repeat_interleave(S), pad["vmin"].repeat_interleave(S), pad["step"].repeat_interleave(S),
        pad["gs"].repeat_interleave(S), pad["ls"].repeat_interleave(S), pad["ir"].repeat_interleave(S),
        P, N, device, gen)
    pf = pf.view(W, S, -1).double().cpu().numpy(); ps = ps.view(W, S, -1).double().cpu().numpy()
    ll = ll.view(W, S).cpu().numpy()
    out_f = []; out_s = []
    for w in range(W):
        T = len(inps[w]["md"]); llw = ll[w]
        if np.isfinite(llw).any():
            mx = np.nanmax(llw[np.isfinite(llw)]); lk = np.where(np.isfinite(llw), llw-mx, -np.inf)
            ww = np.exp(lk / max(P["likelihood_scale"], 1e-6)); s = ww.sum()
            ww = ww/s if s > 1e-300 else np.full(S, 1.0/S)
        else:
            ww = np.full(S, 1.0/S)
        out_f.append((ww[:, None]*pf[w, :, :T]).sum(0))
        out_s.append((ww[:, None]*ps[w, :, :T]).sum(0))
    if device == "cuda": torch.cuda.synchronize()
    return out_f, out_s

# --- v27: valid坑井の PF入力 + 尤度grid(typewell / 混合) ---
def build_ctx(va, tw, P=P0):
    md = va["md"]; z = va["z"]; ev = va["eval_mask"]; kn = ~ev
    tw = tw.sort_values("TVT"); tw_tvt = tw["TVT"].to_numpy(float); tw_gr = tw["GR"].to_numpy(float)
    fb = float(np.nanmean(tw_gr)) if np.isfinite(np.nanmean(tw_gr)) else 0.0
    tw_gr_pf = _smooth(tw_gr, fb, P["tw_gr_smooth_r"]).astype(np.float64)
    gg, gmin, gst = _grid(tw_tvt, tw_gr_pf)
    gr_sm = _smooth(va["gr"], fb, P["hgr_smooth_r"])
    ki = np.where(kn)[0]
    if len(ki) >= 20:
        resid = gr_sm[ki] - np.interp(va["tvt_input"][ki], tw_tvt, tw_gr_pf); gs = float(np.nanstd(resid))
        if not np.isfinite(gs) or gs <= 0: gs = P["gr_sig_def"]
    else:
        gs = P["gr_sig_def"]
    gs = float(np.clip(gs * P["gr_sig_mult"], P["gr_sig_min"], max(P["gr_sig_max"], P["gr_sig_min"] + 1e-6)))
    ls = float(va["tvt_input"][ki[-1]] + z[ki[-1]])
    tl = ki[-30:]; dt = np.diff(va["tvt_input"][tl]); dz = np.diff(z[tl]); dm = np.diff(md[tl]); mm = dm > 0
    ir = float(np.median((dt + dz)[mm] / dm[mm])) if int(mm.sum()) >= 3 else 0.0
    return dict(md=md[ev].astype(np.float64), z=z[ev].astype(np.float64), gr=gr_sm[ev].astype(np.float64),
                gs=gs, ls=ls, ir=ir, gmin=float(gmin), gst=float(gst), gg_tw=np.asarray(gg, np.float64),
                kn_tvt=va["tvt_input"][ki].astype(np.float64), kn_gr=gr_sm[ki].astype(np.float64))  # best1用

def to_inp(ctx, gg, N):
    return dict(md=ctx["md"], z=ctx["z"], gr=ctx["gr"], gg=np.asarray(gg, np.float64),
                gmin=ctx["gmin"], gst=ctx["gst"], gs=ctx["gs"], ls=ctx["ls"], ir=ctx["ir"], N=N)

def _ref_pseudo(r, gmin, step, G):   # 参照1本を疑似typewell(TVTビン平均GR)に
    rt = r["tvt"]; rg = _smooth(r["gr"], float(np.nanmean(r["gr"])), P0["hgr_smooth_r"]).astype(float)
    m = np.isfinite(rt) & np.isfinite(rg); ii = np.round((rt[m] - gmin) / step).astype(int)
    ok = (ii >= 0) & (ii < G); ii, v = ii[ok], rg[m][ok]
    s = np.bincount(ii, weights=v, minlength=G); c = np.bincount(ii, minlength=G)
    return np.where(c > 0, s / np.maximum(c, 1), np.nan)

def mixed_grid(ctx, refs, w_hw=MIX_W, pool=POOL_MODE):
    gg = ctx["gg_tw"].copy(); gmin = ctx["gmin"]; step = ctx["gst"]; G = len(gg)
    # best1: valid の既知prefix GR を最も再現する参照1本を選ぶ(spatialでなくGR-fit)
    if pool == "best1" and len(refs) > 1:
        kb = np.round((ctx["kn_tvt"] - gmin) / step).astype(int); vk = (kb >= 0) & (kb < G)
        best, bres = refs[0], np.inf
        for r in refs:
            gh = _ref_pseudo(r, gmin, step, G); pred = gh[kb[vk]]
            res = ctx["kn_gr"][vk] - pred; res = res[np.isfinite(res)]
            rr = float(np.sqrt(np.mean(res * res))) if len(res) else np.inf
            if rr < bres: bres, best = rr, r
        refs = [best]
    rt = np.concatenate([r["tvt"] for r in refs])
    rg = np.concatenate([_smooth(r["gr"], float(np.nanmean(r["gr"])), P0["hgr_smooth_r"]).astype(float) for r in refs])
    m = np.isfinite(rt) & np.isfinite(rg); rt, rg = rt[m], rg[m]
    idx = np.round((rt - gmin) / step).astype(int); ok = (idx >= 0) & (idx < G); idx, rgv = idx[ok], rg[ok]
    if pool == "median":
        ghw = np.full(G, np.nan); med = pd.Series(rgv).groupby(idx).median()
        ghw[med.index.to_numpy()] = med.to_numpy()
    else:  # mean / best1(単一参照)
        sums = np.bincount(idx, weights=rgv, minlength=G); cnts = np.bincount(idx, minlength=G)
        ghw = np.where(cnts > 0, sums / np.maximum(cnts, 1), np.nan)
    have = np.isfinite(ghw); gm = gg.copy(); gm[have] = w_hw * ghw[have] + (1 - w_hw) * gg[have]
    return gm

def run_batched(inps, P=P0, chunk=CHUNK):
    outs = [None] * len(inps)
    for c0 in range(0, len(inps), chunk):
        _, os_ = run_smoother_ens(inps[c0:c0 + chunk], P, V27_DEV, SEED, N_SEED)
        for j, o in enumerate(os_): outs[c0 + j] = o
        if V27_DEV == "cuda": torch.cuda.empty_cache()
    return outs
print("PF engine (v25 r0 torch/GPU) ready | N=%d n_seed=%d lag=%d mix_w=%.2f" % (PF_N, N_SEED, SMOOTH_LAG, MIX_W))

In [ ]:
# ===== PF A/B (GPUバッチ): typewellのみ vs +疑似typewell混合 =====
PF_MAX_WELLS = None     # 動作確認は 6 等に絞る
_pairs = oof_pairs if PF_MAX_WELLS is None else oof_pairs[:PF_MAX_WELLS]
va_list   = [valid_arrays(p["valid"]) for p in _pairs]
refs_list = [[ref_arrays(r) for r in p["refs"]] for p in _pairs]
true_list = [va["tvt_true"][va["eval_mask"]] for va in va_list]
cov_list  = []
for i in range(len(_pairs)):
    a, b = _ref_pool_tvt(refs_list[i]); cov_list.append((true_list[i] >= a) & (true_list[i] <= b))
ctxs = [build_ctx(va_list[i], TW_CACHE[_pairs[i]["valid"]], P0) for i in range(len(_pairs))]
inps_tw  = [to_inp(ctxs[i], ctxs[i]["gg_tw"], PF_N) for i in range(len(_pairs))]
inps_mix = [to_inp(ctxs[i], mixed_grid(ctxs[i], refs_list[i], MIX_W), PF_N) for i in range(len(_pairs))]

t0 = time.time()
out_tw  = run_batched(inps_tw,  P0)
out_mix = run_batched(inps_mix, P0)
print(f"PF done {time.time()-t0:.0f}s on {V27_DEV} (wells={len(_pairs)}, n_seed={N_SEED}, lag={SMOOTH_LAG})")

preds = {
    "hold":              [np.full(len(true_list[i]), va_list[i]["last_known_tvt"]) for i in range(len(_pairs))],
    "PF typewell":       [np.asarray(out_tw[i],  float)[:len(true_list[i])] for i in range(len(_pairs))],
    "PF +混合(%g)" % MIX_W: [np.asarray(out_mix[i], float)[:len(true_list[i])] for i in range(len(_pairs))],
    "ORACLE上限":        [np.where(cov_list[i], true_list[i], va_list[i]["last_known_tvt"]) for i in range(len(_pairs))],
}
# ゲート: 参照本数<=MIX_MAX_REFS の坑井だけ混合、それ以外は typewell単体(追加PF不要, 既存出力を選択)
_gate = [out_mix[i] if len(refs_list[i]) <= MIX_MAX_REFS else out_tw[i] for i in range(len(_pairs))]
preds["PF 混合gated(<=%dref)" % MIX_MAX_REFS] = [np.asarray(_gate[i], float)[:len(true_list[i])] for i in range(len(_pairs))]
def _pool(name, idxs):
    E = np.concatenate([preds[name][i] - true_list[i] for i in idxs])
    C = np.concatenate([cov_list[i] for i in idxs])
    r = lambda x: float(np.sqrt(np.mean(x*x))) if len(x) else float("nan")
    return r(E), r(E[C]), r(E[~C]), float(C.mean())
subsets = [("全", list(range(len(_pairs)))),
           ("参照1本", [i for i, p in enumerate(_pairs) if p["n_refs"] == 1]),
           ("参照2-5本", [i for i, p in enumerate(_pairs) if p["n_refs"] >= 2])]
for sname, idxs in subsets:
    if not idxs: continue
    print(f"[{sname}] valid {len(idxs)}本   {'overall':>8s} {'covered':>8s} {'toe':>8s} {'cov%':>6s}")
    for name in preds:
        o, c, t, cf = _pool(name, idxs)
        print(f"  {name:16s} {o:8.3f} {c:8.3f} {t:8.3f} {cf*100:5.1f}")
    print()
print("※ PF=v25 r0 torch(GPU, 尤度加重seed, 固定ラグ%d)。grid以外はv25と完全同一。" % SMOOTH_LAG)
print("※ 設定: MIX_W=%.2f POOL_MODE=%s。参照多数群は mean=平均ボケ→median/best1 を試す。0=typewellのみ。" % (MIX_W, POOL_MODE))

In [ ]:
# ===== 追加: PF再実行なしで混ぜ方を探る(gridのMIX_Wとは別物) =====
# ※grid の MIX_W を変える=尤度が変わりPF軌道が変わる=本来は要PF再実行。
#   ここは保存済み out_tw(MIX0) / out_mix(MIX1) の後段処理なので再実行不要の近似。
TW = [np.asarray(out_tw[i], float)[:len(true_list[i])] for i in range(len(_pairs))]
MX = [np.asarray(out_mix[i], float)[:len(true_list[i])] for i in range(len(_pairs))]
_ALL = list(range(len(_pairs)))
def _pool3(pl, idxs):
    E = np.concatenate([pl[i] - true_list[i] for i in idxs]); C = np.concatenate([cov_list[i] for i in idxs])
    r = lambda x: float(np.sqrt(np.mean(x*x))) if len(x) else float("nan")
    return r(E), r(E[C]), r(E[~C])
_subs = [("全", _ALL),
         ("参照1本", [i for i, p in enumerate(_pairs) if p.get("n_refs", 1) == 1]),
         ("参照2-5本", [i for i, p in enumerate(_pairs) if p.get("n_refs", 1) >= 2])]

print("=== 出力ブレンド a: pred=(1-a)*tw + a*mix (PF再実行なし) ===")
for sname, idxs in _subs:
    if not idxs: continue
    row = []
    for a in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]:
        pl = [(1 - a) * TW[i] + a * MX[i] for i in _ALL]
        row.append((a, _pool3(pl, idxs)[0]))
    best = min(row, key=lambda x: x[1])
    print(f"[{sname:8s}] " + " ".join("a%.1f=%.3f" % (a, o) for a, o in row) + f"  ★best a={best[0]} ({best[1]:.3f})")

print("\n=== 行別コンポジット covered=tw / toe=mix (PF再実行なし) ===")
_comp = [np.where(cov_list[i], TW[i], MX[i]) for i in _ALL]
for sname, idxs in _subs:
    if not idxs: continue
    o, c, t = _pool3(_comp, idxs); otw = _pool3(TW, idxs)[0]; omx = _pool3(MX, idxs)[0]
    print(f"[{sname:8s}] composite={o:.3f} (cov {c:.3f}, toe {t:.3f}) | tw {otw:.3f} / mix {omx:.3f}")
print("\n※ 真の grid-MIX_W 掃引(尤度を変える)は PF 再実行が必要。上は tw/mix 出力の後段近似。")

In [ ]:
# ===== n_refs 別(2/3/4/5本 個別)に出力ブレンド & コンポジット(PF再実行なし) =====
_nvals = sorted(set(p.get("n_refs", 1) for p in _pairs))
_subs2 = [("全", _ALL)] + [("参照%d本" % k, [i for i, p in enumerate(_pairs) if p.get("n_refs", 1) == k]) for k in _nvals]

print("=== 出力ブレンド a: (1-a)*tw + a*mix ===")
for sname, idxs in _subs2:
    if not idxs: continue
    row = []
    for a in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]:
        pl = [(1 - a) * TW[i] + a * MX[i] for i in _ALL]
        row.append((a, _pool3(pl, idxs)[0]))
    best = min(row, key=lambda x: x[1])
    print(f"[{sname:9s} n={len(idxs):3d}] " + " ".join("a%.1f=%.3f" % (a, o) for a, o in row) + f"  ★a={best[0]}({best[1]:.3f})")

print("\n=== covered=tw / toe=mix コンポジット ===")
_comp = [np.where(cov_list[i], TW[i], MX[i]) for i in _ALL]
for sname, idxs in _subs2:
    if not idxs: continue
    o, c, t = _pool3(_comp, idxs); otw = _pool3(TW, idxs)[0]; omx = _pool3(MX, idxs)[0]
    print(f"[{sname:9s} n={len(idxs):3d}] composite={o:.3f} (cov {c:.3f}, toe {t:.3f}) | tw {otw:.3f} / mix {omx:.3f}")

In [ ]:
# ===== 最近傍距離バケット別に 出力ブレンド a を掃引(PF再実行なし) =====
_edges = [0, 250, 500, 1000, 1500, 2000, 3000, 5000, 1e9]   # nn_dist(最近傍距離)の境界
def _bkt(d):
    for k in range(len(_edges) - 1):
        if _edges[k] <= d < _edges[k + 1]: return k
    return len(_edges) - 2
_nnb = [_bkt(_pairs[i].get("nn_dist", 0.0)) for i in _ALL]

print("=== 最近傍距離バケット別 出力ブレンド a: (1-a)*tw + a*mix (a0=tw, a1=mix) ===")
for k in range(len(_edges) - 1):
    idxs = [i for i in _ALL if _nnb[i] == k]
    if not idxs: continue
    row = []
    for a in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.7, 1.0]:
        pl = [(1 - a) * TW[i] + a * MX[i] for i in _ALL]
        row.append((a, _pool3(pl, idxs)[0]))
    best = min(row, key=lambda x: x[1])
    lbl = "%d-%d" % (_edges[k], _edges[k + 1]) if _edges[k + 1] < 1e9 else "%d+" % _edges[k]
    print(f"[nn {lbl:>10s} n={len(idxs):4d}] " + " ".join("a%.1f=%.3f" % (a, o) for a, o in row) + f"  ★a={best[0]}({best[1]:.3f})")

In [ ]:
# ===== 距離連動α: 最近傍が近いほど α大 (PF再実行なし) =====
def _alpha(d):
    # 制御点(m, α): (250,0.7)(500,0.6)(1000,0.45)(1500,0.3)。<=250は0.7, >1500は0(混ぜない=先端)
    if d <= 250: return 0.7
    if d > 1500: return 0.0
    return float(np.interp(d, [250, 500, 1000, 1500], [0.7, 0.6, 0.45, 0.3]))

def _dist_pred(need_refs):
    out = []
    for i in _ALL:
        d = _pairs[i].get("nn_dist", 1e9); nr = _pairs[i].get("n_refs", 1)
        a = _alpha(d) if nr >= need_refs else 0.0
        out.append((1 - a) * TW[i] + a * MX[i])
    return out

otw = _pool3(TW, _ALL)[0]
of4 = _pool3([(0.6) * TW[i] + 0.4 * MX[i] for i in _ALL], _ALL)[0]
print("=== 距離連動α (250以下0.7 / 500:0.6 / 1000:0.45 / 1500:0.3 / >1500:混ぜない) ===")
print("  tw単体      : %.3f" % otw)
print("  固定 a=0.4  : %.3f (%+.3f)" % (of4, otw - of4))
for nr in [1, 2]:
    pred = _dist_pred(nr)
    o, c, t = _pool3(pred, _ALL)
    nmix = sum(1 for i in _ALL if (_pairs[i].get("n_refs",1) >= nr and _alpha(_pairs[i].get("nn_dist",1e9)) > 0))
    tag = "距離α(全)" if nr == 1 else "距離α+参照>=2"
    print("  %-14s: %.3f (%+.3f)  covered %.3f toe %.3f | 混合%d坑井" % (tag, o, otw - o, c, t, nmix))

In [ ]:
# ===== 距離連動α を 参照本数(n_refs)別に見る (PF再実行なし) =====
_pred_da = _dist_pred(1)   # 距離αのみ(本数ゲート無し)
_nvals = sorted(set(p.get("n_refs", 1) for p in _pairs))
print("=== 距離連動α の n_refs 別 (tw vs 距離α) ===")
for k in _nvals:
    idxs = [i for i in _ALL if _pairs[i].get("n_refs", 1) == k]
    if not idxs: continue
    o_da = _pool3(_pred_da, idxs)[0]; o_tw = _pool3(TW, idxs)[0]; o_mx = _pool3(MX, idxs)[0]
    a_mean = np.mean([_alpha(_pairs[i].get("nn_dist", 1e9)) for i in idxs])
    print("[参照%d本 n=%3d] tw=%.3f  距離α=%.3f (%+.3f)  mix=%.3f | 平均α=%.2f" % (
        k, len(idxs), o_tw, o_da, o_tw - o_da, o_mx, a_mean))